Teniendo en cuenta la elevada frecuencia de lectura de los endpoints (`GET /dungeon` con 1M/día, `GET /room` con 500K/día, `GET /user` con 300K/día) frente a una escritura moderada (10K comentarios/día) y una operación de borrado casi anecdótica (100 monstruos/año), se pueden aplicar los siguientes patrones de diseño para optimizar el rendimiento y la escalabilidad del prototipo en MongoDB.

### 1. **Patrón Subset** (Subconjunto)

**Problema que soluciona:**  
Documentos excesivamente grandes que incluyen datos de acceso poco frecuente, ocupando memoria RAM y perjudicando las lecturas de los campos más consultados.

**Mejora propuesta:**

- **Colección `Rooms`**:  
  Actualmente el array `hints` almacena **todos** los comentarios de una sala. El endpoint `GET /room/{room_id}` solo necesita los últimos 20 comentarios.  
  Por tanto, es mejor eliminar el array completo de la sala y, si se desea máxima velocidad, mantener solo un subconjunto con los 20 comentarios más recientes. El historial completo se mueve a una colección independiente `Hints`.  
  Con esto la sala ocupa mucho menos espacio y las lecturas frecuentes (500K/día) se sirven desde un solo documento más ligero.

  Independiente de que tengamos que escribir en `Hints` cada vez que se añade un comentario, el volumen de escritura (10K/día) es manejable y no afecta significativamente el rendimiento.

- **Colección `Users`**:  
  El endpoint `GET /user/{email}` (300K/día) devuelve los últimos 20 comentarios del usuario. Si los comentarios solo residieran en `Hints`, habría que hacer una consulta con `$lookup` o un `find` sobre `Hints` ordenado y limitado, lo que añade latencia.  
  Por ello, es mejor mantener un array `last_20_comments` en el documento del usuario, que se actualiza cada vez que se publica un nuevo comentario (usando `$push` con `$slice` para conservar solo los 20 más recientes). Así la lectura de usuario es directa y no requiere acceder a otra colección.  
  El subconjunto se sincroniza en las escrituras, cuyo volumen (10K/día) es perfectamente asumible.

### 2. **Patrón Computed** (Valores precalculados)

**Problema que soluciona:**  
Consultas repetitivas que requieren agregaciones o cálculos costosos sobre datos que cambian con poca frecuencia, penalizando las lecturas masivas.

**Mejora propuesta:**

- **Endpoint `GET /room/{room_id}`** (500K/día):  
  Devuelve el número de monstruos de cada tipo y el total de oro de los tesoros de la sala.  
  Podríamos añadir campos precalculados como `total_gold` y `monster_count_by_type` en el documento de la sala. Se actualizan únicamente cuando se modifica la composición de monstruos/tesoros de la sala (operación rarísima), evitando recorrer arrays anidados en cada lectura.

- **Endpoint `GET /dungeon/{dungeon_id}`** (1M/día):  
  Necesita el número de comentarios de cada categoría para cada sala de la mazmorra.  
  Parecido al anterior, podemos almacenar en el documento de cada sala (o en el de la mazmorra) un objeto `comments_by_category` (p. ej. `{bug: 34, hint: 12, suggestion: 5, lore: 8}`). Este contador se incrementa atómicamente con cada `POST /comment` (solo 10K/día), de modo que la consulta de mazmorra no tenga que agregar sobre la colección `Hints` en tiempo real.

Es mejor calcular una vez al escribir que repetir el cálculo millones de veces al leer (en este caso, como el ratio lectura/escritura es de 100:1, el beneficio es enorme).


### 3. **Outlier** (Datos atípicos)

**Problema que soluciona:**  
Dentro de cada room, hay algunos que tienen un número elevado de loot, mientras hay otros que no tienen ninguno. Esto hace que el documento de la room sea muy grande, lo que afecta a las lecturas frecuentes.

**Mejora propuesta:**
Podemos separar los loot en una colección independiente `Loot`, donde cada documento representa un loot individual asociado a una room. De esta forma, el documento de la room se mantiene ligero y las lecturas de la room no se ven afectadas por el número de loot. Para obtener los loot de una room, se puede hacer una consulta específica a la colección `Loot` filtrando por `room_id`. Dado que el número de loot por room es variable, esta separación permite manejar eficientemente tanto las rooms con muchos loot como las que no tienen ninguno, sin penalizar el rendimiento de las lecturas frecuentes de las rooms.

Por supuesto, tendríamos que antes ponerle el flag. 


### 4. **Justificación global**

La combinación de estos tres patrones permite que:

- Las lecturas más frecuentes (`dungeon`, `room`, `user`) se sirvan desde uno o dos documentos como máximo, sin agregaciones costosas en el motor de base de datos.
- El único punto con escrituras significativas (`POST /comment`) actualiza de forma controlada los subconjuntos y los contadores precalculados, aprovechando operaciones atómicas.
- Creamos un atribuuto nuevo que marque flag un documento como outlier, lo que nos permite tratarlo de forma diferente (p. ej. almacenarlo en una colección aparte) para evitar que afecte al rendimiento de las lecturas frecuentes.